In [30]:
import pandas as pd
import re #importa expressões regulares(regex)

In [31]:
df_2025 = pd.read_csv("Clockify_Time_Report_Detailed_01_01_2025-31_12_2025.csv")
df_2026 = pd.read_csv("Clockify_Time_Report_Detailed_01_01_2026-31_12_2026.csv")

#Compara se os dois dataframes tem as mesmas colunas, se não retorna erro
assert list(df_2025.columns) == list(df_2026.columns), "Colunas diferentes entre os arquivos!"

In [32]:
#Concatena os dois dataframes, ignore_index ignora indices originais e recria-os
df = pd.concat([df_2025, df_2026], ignore_index=True)
#Verifica se o tamanho do df novo bate com a soma dos dois originais
assert len(df) == len(df_2025) + len(df_2026), "Tamanho do dataframe final não bate com a soma dos dois originais!"

In [33]:
#Remove dados irrelevantes
tags_fora_escopo = ['Suporte','Reunião']

df = df[(df['Tags'].notna()) & (~df['Tags'].isin(tags_fora_escopo))].reset_index(drop=True)

In [ ]:
#Extrai o número do chamado e o restante da descrição
df[['numero_chamado','descricao_resto']] = df['Description'].str.extract(r'#(\d+)\s*(.*)')

In [35]:
#Criando numero de demanda para colunas sem
sem_numero = df['numero_chamado'].isna()

chave = df.loc[sem_numero, ['Client', 'Project', 'Description']].fillna('').astype(str).agg(' | '.join, axis=1)

codigos, _ = pd.factorize(chave)
df.loc[sem_numero, 'numero_chamado'] = [f'S{c+1:04d}' for c in codigos]  

df['descricao_resto'] = df['descricao_resto'].fillna(df['Description'])

In [36]:
#Cria a mascara para os clientes, sistemas e chamados
def montar_mapa(serie, prefixo):
    valores_unicos = sorted(serie.dropna().unique())
    return {valor: f"{prefixo}_{i+1:03d}" for i, valor in enumerate(valores_unicos)}

mapa_clientes  = montar_mapa(df['Client'], 'Cliente')
mapa_sistemas  = montar_mapa(df['Project'], 'Sistema')
mapa_chamados  = montar_mapa(df['numero_chamado'], 'Chamado')

In [37]:
#Cria o dataframe mascarado com as colunas originais e as mascaradas
df['cliente_mascarado'] = df['Client'].map(mapa_clientes)
df['sistema_mascarado'] = df['Project'].map(mapa_sistemas)
df['chamado_mascarado'] = df['numero_chamado'].map(mapa_chamados)

In [38]:
#Criando nova coluna com descrição do chamado mascarada
df['descricao_mascarada'] = (df['chamado_mascarado'] + ' - ' + df['descricao_resto']).fillna(df['Description'])

In [39]:
#Substitui nome dos clientes nas descrições
for real, codigo in sorted(mapa_clientes.items(),key=lambda x: -len(x[0])):
    df['descricao_mascarada'] = df['descricao_mascarada'].str.replace(
        re.escape(real), codigo, case=False, regex=True
    )

In [40]:
nomes_extras = ['ELTON', 'Felipe', 'GRUPO CCA', 'Indipeso', 'JVA', 'Replast']

mapa_extra = montar_mapa(pd.Series(nomes_extras), 'Empresa')

for real,codigo in sorted(mapa_extra.items(), key=lambda x: -len(x[0])):
    df['descricao_mascarada'] = df['descricao_mascarada'].str.replace(
        r'\b' + re.escape(real) + r'\b', codigo, case=False , regex=True    
    )

In [41]:
#Dropa as colunas originais e as que não vão ser úteis
df_mascarado = df.drop(columns=['Client', 'Project', 'Description', 'descricao_resto','numero_chamado', 'Email', 'Task', 'Group'])

In [42]:
# Cria o csv com os dados mascarados
df_mascarado.to_csv("clockify_unificado_mascarado.csv", index=False)

In [43]:
# Cria o csv com um mapa de dados privados para referência
mapa_df = pd.concat([
    pd.DataFrame(mapa_clientes.items(), columns=['valor_real', 'valor_mascarado']).assign(tipo='cliente'),
    pd.DataFrame(mapa_sistemas.items(), columns=['valor_real', 'valor_mascarado']).assign(tipo='sistema'),
    pd.DataFrame(mapa_chamados.items(), columns=['valor_real', 'valor_mascarado']).assign(tipo='chamado'),
    pd.DataFrame(mapa_extra.items(), columns=['valor_real', 'valor_mascarado']).assign(tipo='Empresa'),
], ignore_index=True)

mapa_df.to_csv("mapeamento_privado.csv", index=False)

In [46]:
#Verifica se ainda tem dados privados
for real in list(mapa_clientes) + list(mapa_extra):
    qtd = df['descricao_mascarada'].str.contains(re.escape(real), case=False, na=False).sum()
    if qtd > 0:
        print(f"AINDA APARECE: '{real}' em {qtd} linhas")

In [47]:
df[df['numero_chamado'].str.startswith('S', na=False)].groupby('numero_chamado').agg(
    sessoes=('Duration (decimal)', 'size'),
    horas=('Duration (decimal)', 'sum'),
    desc=('Description', 'first')
).sort_values('horas', ascending=False).head(15)


,sessoes,horas,desc
numero_chamado,,,
S0030,70,87.52,Projeto IBS BI
S0038,13,14.61,Alterações Power Bi Suporte
S0024,15,12.81,Refinando front com impeccable - IBS Bi
S0010,7,11.97,"Correção Curva ABC, roteamento e previsão vs r..."
S0023,8,9.18,Adicionando função de biblioteca - IBS Bi
S0037,4,8.13,Divergência entre BI e WBI
S0007,7,8.05,Padronizei resolução dinâmica de colunas (role...
S0032,4,7.80,Adicionar informações do mural no power bi do ...
S0013,6,6.73,"Padroniza respostas do chat (margem, faturamen..."
